# Titanic Dataset Analysis – AI Assignment
### Data Cleaning, Feature Engineering, and Feature Selection

**Student:** ____________________  
**Course:** Artificial Intelligence  
**Date:** 2026-03-08

This notebook performs:
- Data exploration
- Data cleaning
- Feature engineering
- Feature selection
- Visual analysis

Dataset: Titanic – Machine Learning from Disaster.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier

pd.set_option('display.max_columns', None)

## 2. Load Dataset

In [ ]:
df = pd.read_csv('../data/train.csv')
df.head()

## 3. Dataset Overview

In [ ]:
df.info()

In [ ]:
df.describe()

## 4. Missing Values

In [ ]:
df.isnull().sum()

## 5. Data Cleaning

In [ ]:
# Fill missing Age with median
df['Age'] = df['Age'].fillna(df['Age'].median())

# Fill Embarked with mode
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

# Fill Fare with median
df['Fare'] = df['Fare'].fillna(df['Fare'].median())

# Cabin missing values
df['Cabin'] = df['Cabin'].fillna('Unknown')

# Standardize Sex column
df['Sex'] = df['Sex'].str.lower()

# Remove duplicates
df = df.drop_duplicates()

df.isnull().sum()

## 6. Outlier Detection

In [ ]:
sns.boxplot(x=df['Fare'])
plt.title('Fare Outliers')
plt.show()

In [ ]:
upper = df['Fare'].quantile(0.99)
df['Fare'] = np.where(df['Fare'] > upper, upper, df['Fare'])

## 7. Feature Engineering

In [ ]:
# Family size
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

# IsAlone
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

In [ ]:
# Title extraction
df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\\.', expand=False)

In [ ]:
# Deck extraction
df['Deck'] = df['Cabin'].str[0]

In [ ]:
# Age groups
def age_group(age):
    if age < 13:
        return 'Child'
    elif age < 20:
        return 'Teen'
    elif age < 60:
        return 'Adult'
    else:
        return 'Senior'

df['AgeGroup'] = df['Age'].apply(age_group)

In [ ]:
# Fare per person
df['FarePerPerson'] = df['Fare'] / df['FamilySize']

## 8. Feature Transformation

In [ ]:
df['Fare_log'] = np.log1p(df['Fare'])

## 9. Categorical Encoding

In [ ]:
df = pd.get_dummies(df, columns=['Sex','Embarked','Title','Deck'], drop_first=True)

## 10. Survival Visualization

In [ ]:
sns.countplot(x='Survived', data=df)
plt.title('Survival Distribution')
plt.show()

## 11. Feature Selection using Random Forest

In [ ]:
drop_cols = ['Name','Ticket','Cabin','AgeGroup']
df_model = df.drop(columns=drop_cols, errors='ignore')

X = df_model.drop('Survived', axis=1)
y = df_model['Survived']

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X, y)

importance = pd.Series(model.feature_importances_, index=X.columns)
importance.sort_values(ascending=False).head(10)

## 12. Key Observations

- Female passengers had higher survival rates.
- First class passengers survived more than third class.
- Smaller family sizes had slightly higher survival chances.
- Fare and passenger class strongly influenced survival.

These engineered features improve the predictive power of machine learning models.